<a href="https://colab.research.google.com/github/wycliffenguono/BigData/blob/main/ARM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Install Pyspark and Java Dependency

In [ ]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!pip install pyspark -q


# 2. Import relevant modules

In [ ]:
import random
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.types import StructType, StructField, StringType, ArrayType
from pyspark.ml.fpm import FPGrowth

# 3. Initialize Spark Session

*   Setting memory to 10GB to by-pass default 1GB configuration

*   Reducing task partitions to 8, from the default 200 to reduce scheduling overhead





In [ ]:
spark = SparkSession.builder \
        .appName("Simulated_Retail_ARM_Project") \
        .config("spark.driver.memory", "10g") \
        .config("spark.sql.shuffle.partitions", "8") \
        .getOrCreate()

# 4. Define standard retail categories for the simulation

In [ ]:
categories = ["Milk", "Bread", "Eggs", "Butter", "Cheese", "Soda", "Chips",
              "Yogurt", "Apples", "Bananas", "Chicken", "Beef", "Pasta", "Tomato_Sauce"]


# 5. Python Generator to simulate millions of transaction rows efficiently

* We are using 600, 000 baskets to store retail items.
* Each basket is assigned a base random size ranging between 1-4, an average of 2.5
* We conditionally append matching items (e.g., if a basket has Bread, it adds Butter). This pushes the true average closer to 3 to 4 items per basket.
* 600,000 Baskets × 4 average items = approximately 2,400,000 individual item records.
* We use random probability threshold to ensure the simulated shoppers behave like real human beings, for example:
    * 90% of people buying pasta might buy sauce because they are making dinner tonight.
    * The other 10% might skip the sauce because they already have a jar at home, or they prefer making plain noodles.





In [ ]:
def generate_simulated_retail_data(num_baskets=600000):
    start_date = datetime(2026, 1, 1)
    for i in range(1, num_baskets + 1):
        tx_id = f"TX_{i:06d}"

        # Simulating random timestamps across a full trading year
        random_seconds = random.randint(0, 31536000)
        timestamp = (start_date + timedelta(seconds=random_seconds)).strftime("%Y-%m-%d %H:%M:%S")

        # Creating a basket with random base size (1 to 4 items)
        basket = set(random.sample(categories, random.randint(1, 4)))

        # Injected high-probability association patterns
        if "Bread" in basket and random.random() < 0.85: basket.add("Butter")
        if "Pasta" in basket and random.random() < 0.90: basket.add("Tomato_Sauce")
        if "Milk" in basket and random.random() < 0.75: basket.add("Eggs")
        if "Chips" in basket and random.random() < 0.70: basket.add("Soda")

        yield tx_id, list(basket), timestamp

# 6.  Set Schema and build the DataFrame in memory

In [ ]:
schema = StructType([
    StructField("TransactionID", StringType(), False),
    StructField("Product_Categories", ArrayType(StringType()), False),
    StructField("Timestamp", StringType(), False)
])

print("=== Generating Records of Simulated Retail Data ===")
df_simulated = spark.createDataFrame(generate_simulated_retail_data(600000), schema=schema).cache()
print(f"Success! Generated {df_simulated.count()} unique transaction baskets.")

# Show a preview of the simulated data attributes
df_simulated.show(5, truncate=False)

=== Generating Records of Simulated Retail Data ===
Success! Generated 600000 unique transaction baskets.
+-------------+---------------------------------------+-------------------+
|TransactionID|Product_Categories                     |Timestamp          |
+-------------+---------------------------------------+-------------------+
|TX_000001    |[Tomato_Sauce, Milk, Eggs]             |2026-08-11 04:34:23|
|TX_000002    |[Butter, Chicken, Yogurt, Tomato_Sauce]|2026-01-06 06:35:13|
|TX_000003    |[Butter, Chicken]                      |2026-03-10 09:46:45|
|TX_000004    |[Bread, Butter, Bananas]               |2026-08-02 23:31:49|
|TX_000005    |[Yogurt, Milk, Eggs]                   |2026-09-16 01:51:53|
+-------------+---------------------------------------+-------------------+
only showing top 5 rows


# 7. Run the FP-Growth Pipeline

In [ ]:
# Settings: Min Support 2% and Min Confidence 40%
fp_growth = FPGrowth(itemsCol="Product_Categories", minSupport=0.02, minConfidence=0.4)
model = fp_growth.fit(df_simulated)


print("\n=== Key Findings: Most Frequent Itemsets ===")
frequent_itemsets = model.freqItemsets.orderBy(f.desc("freq"))
frequent_itemsets.show(15, truncate=False)


=== Key Findings: Most Frequent Itemsets ===
+---------------------+------+
|items                |freq  |
+---------------------+------+
|[Tomato_Sauce]       |188356|
|[Butter]             |184184|
|[Eggs]               |175322|
|[Soda]               |170353|
|[Milk]               |107447|
|[Bread]              |107356|
|[Chips]              |107336|
|[Chicken]            |107321|
|[Yogurt]             |107305|
|[Beef]               |107272|
|[Bananas]            |107112|
|[Pasta]              |107085|
|[Apples]             |107049|
|[Cheese]             |106779|
|[Pasta, Tomato_Sauce]|98103 |
+---------------------+------+
only showing top 15 rows


# 8. Discover and order Association Rules

In [ ]:
print("\n=== Key Findings: Significant Association Rules ===")

association_rules = model.associationRules \
    .select(
        f.col("antecedent").alias("If Customer Buys"),
        f.col("consequent").alias("They Also Buy"),
        f.round("support", 4).alias("Support"),
        f.round("confidence", 4).alias("Confidence"),
        f.round("lift", 2).alias("Lift")
    ).orderBy(f.desc("confidence"))

association_rules.show(15, truncate=False)


=== Key Findings: Significant Association Rules ===
+----------------------+--------------+-------+----------+----+
|If Customer Buys      |They Also Buy |Support|Confidence|Lift|
+----------------------+--------------+-------+----------+----+
|[Pasta]               |[Tomato_Sauce]|0.1635 |0.9161    |2.92|
|[Apples, Pasta]       |[Tomato_Sauce]|0.025  |0.915     |2.91|
|[Pasta, Yogurt]       |[Tomato_Sauce]|0.0254 |0.9143    |2.91|
|[Pasta, Eggs]         |[Tomato_Sauce]|0.0415 |0.9142    |2.91|
|[Pasta, Chips]        |[Tomato_Sauce]|0.0251 |0.9136    |2.91|
|[Cheese, Pasta]       |[Tomato_Sauce]|0.0249 |0.9134    |2.91|
|[Pasta, Bananas]      |[Tomato_Sauce]|0.025  |0.9133    |2.91|
|[Pasta, Soda]         |[Tomato_Sauce]|0.0405 |0.9131    |2.91|
|[Pasta, Beef]         |[Tomato_Sauce]|0.0249 |0.9129    |2.91|
|[Pasta, Butter]       |[Tomato_Sauce]|0.0438 |0.9129    |2.91|
|[Pasta, Bread]        |[Tomato_Sauce]|0.0252 |0.912     |2.91|
|[Pasta, Chicken]      |[Tomato_Sauce]|0.0252 |0.91

# 9.  Clean Up working memory

In [ ]:
spark.stop()